# W11｜Performance / Load Test / Quality Trade-off｜教学版

**建议学习时间：5h**

这本 Notebook 的定位是“第二教材”，不是代码仓库。  
正确使用方式：

1. 先看课程视频；
2. 回到 Notebook 看“是什么 / 为什么 / 什么时候用”；
3. 带着“你应该观察什么”再运行代码；
4. 先自己做题，再展开参考答案；
5. 最后把同一机制接入 ChatBI。

## Excel 任务表 1:1 对照

| # | 模块 | Excel任务 | 输出/验收 | Notebook处理方式 |
|---:|---|---|---|---|
| 1 | Backend | 按下方课程映射完成 Pool/Load Test/Eval | 重点是 p95 与质量-性能 trade-off | 知识点 + Checkpoint/参考答案 |
| 2 | Agent | 学 Eval Metrics / Robustness | 定义核心质量指标 | 知识点 + Checkpoint/参考答案 |
| 3 | Lab | 做 1/10/50 并发压测 | 记录 latency/error rate | 明确作业 + 参考答案/参考实现 |
| 4 | ChatBI | 跑 Golden Eval + 修 1 个瓶颈 | 形成一页性能与质量结果 | 明确作业 + 参考答案/参考实现 |
| 5 | 复盘 | 写 3 个瓶颈和 1 个改进 | 用数据而不是感觉判断 | 明确作业 + 参考答案/参考实现 |

> **规则**：Task 1-2 是学习任务，用 Checkpoint/面试题验收；Task 3-5 是作业任务，下面给明确参考答案/参考实现。设计题没有唯一答案时给参考图/评分标准。

## Excel 作业参考答案 / 验收标准

### Task 3｜1/10/50 并发
分别记录 p50/p95/error rate/throughput，不只看平均耗时。

### Task 4｜Before/After
只改一个变量后比较性能与质量；例如并行独立 Tool、调整合理 pool/semaphore、缓存稳定 metadata。

### Task 5｜参考写法
瓶颈必须引用数据，例如：LLM 占 65% latency、DB pool wait 拉高 p95、独立 Tool 串行；改进也要用 before/after 证明。

> 建议：先完成作业，再对照参考答案。不要直接照抄。

## 本周学习目标

        - 理解 p50/p95/throughput
- 能设计基础 load test
- 能讨论性能-质量-成本权衡

        ## 本周前置检查

        - 完成 W10；有 trace 和 golden metrics

        ## 本周必须记住的英文术语

        - `p95`：95分位延迟
- `throughput`：吞吐量
- `load test`：负载测试
- `bottleneck`：瓶颈
- `trade-off`：权衡

        > 可以听中文翻译，但这些英文术语一定要保留，因为代码、日志、文档和面试都会使用英文。

## 1. 为什么平均延迟不够？

    **是什么（What）**  
    平均值会被大量快请求稀释；p95 表示 95% 请求都不超过这个延迟，更能看到尾部体验。

    **为什么需要（Why）**  
    Production 用户常常抱怨的是 tail latency。

    **什么时候用（When）**  
    API/LLM/tool/DB 性能监控。

    **常见误区 / 容易踩坑**  
    p95 不是最大值，也不是 95% 请求平均值。


**中英术语**：`p50` = 50分位；`p95` = 95分位；`throughput` = 吞吐量；`error rate` = 错误率；`tail latency` = 尾延迟

### 代码实验：计算 p50/p95 与 throughput

**这段代码想让你看什么？**

观察 mean 与 p95 的差别。

> 不要先看代码细节。先带着上面的观察目标运行，再回头逐行读。

In [ ]:
import random, statistics, math

latencies = [random.uniform(50, 120) for _ in range(95)] + [400, 500, 700, 900, 1200]
latencies = sorted(latencies)

def percentile(xs, p):
    k = (len(xs)-1) * p
    f = math.floor(k)
    c = math.ceil(k)
    if f == c:
        return xs[f]
    return xs[f] * (c-k) + xs[c] * (k-f)

print("mean:", round(statistics.mean(latencies), 1))
print("p50:", round(percentile(latencies, .50), 1))
print("p95:", round(percentile(latencies, .95), 1))

## 2. 性能与 AI 质量通常有 Trade-off

    **是什么（What）**  
    更多 LLM/Tool 调用可能提升质量，但增加 latency 和 cost。

    **为什么需要（Why）**  
    Production 优化不能只追求 accuracy，也不能只追求速度。

    **什么时候用（When）**  
    Agent 反思、多检索、多模型调用。

    **常见误区 / 容易踩坑**  
    不要用单一指标决定版本好坏。


**中英术语**：`trade-off` = 权衡；`throughput` = 吞吐量；`cost per request` = 单请求成本

In [ ]:
configs = [
    {"name":"fast", "accuracy":0.78, "p95_ms":800, "cost":0.002},
    {"name":"rerank", "accuracy":0.88, "p95_ms":1400, "cost":0.004},
    {"name":"deep", "accuracy":0.92, "p95_ms":3200, "cost":0.011},
]
for c in configs:
    print(c)

### Checkpoint

1. DB pool 从 10 改成 100 为什么不一定更快？
2. 为什么优化前必须先 measure？
3. 质量提高 2%，p95 翻 3 倍是否值得？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** 数据库本身资源/锁/IO有限，更多连接会增加竞争，甚至让整体吞吐下降。

**2.** 没有基线就不知道瓶颈在哪，也无法证明改动有效。

**3.** 没有统一答案，要看业务价值、SLO、成本和用户场景；这就是工程 trade-off。

    </details>

### 练习：设计一张性能质量报告

列出至少 6 个字段，要求同时覆盖系统性能与 AI 质量。

<details>
    <summary><b>参考答案</b></summary>

    ```python
# 推荐字段：
# config
# p50_ms
# p95_ms
# throughput_rps
# error_rate
# intent_accuracy
# sql_execution_success
# cost_per_request
```

    </details>

## ChatBI 本周作业

        对 ChatBI 做 1 / 10 / 50 并发测试，输出：

`p50 | p95 | throughput | error rate | intent accuracy | cost`

### 本周面试 / 理解检查

1. p95 是什么？
2. 吞吐量和延迟有什么关系？
3. 如何判断瓶颈在 LLM 还是 DB？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** 95% 的观测值小于等于该值，用于描述尾部延迟。

**2.** 在容量内可能共同改善，但接近饱和后增加并发常导致排队和尾延迟上升。

**3.** 用分步骤 trace/span latency、资源指标和受控实验比较。

    </details>

## 本周结束前的自检

只有下面 4 项都能做到，才算真正完成：

- [ ] 我能不用课程原话，用自己的话解释核心概念；
- [ ] 我知道这周的代码实验到底在证明什么；
- [ ] 我至少独立完成了一道代码题；
- [ ] 我能说出一个“生产环境里如果做错会发生什么”的例子。